In [ ]:
#Set the env
import pandas as pd
import openpyxl as pyxl

In [5]:
import os
from pathlib import Path

# 1. Détection dynamique de la racine du projet
# Si ton notebook est dans un sous-dossier (ex: /notebooks), on remonte d'un cran.
# Sinon, on prend le dossier actuel.
BASE_DIR = Path(os.getcwd())
if BASE_DIR.name == 'notebooks':
    PROJECT_ROOT = BASE_DIR.parent
else:
    PROJECT_ROOT = BASE_DIR

# 2. Définition des chemins clés (propres et flexibles)
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "online_retail_II.xlsx"

# 3. Création automatique du dossier data s'il n'existe pas
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Racine du projet : {PROJECT_ROOT}")
print(f"📊 Chemin des données : {RAW_DATA_PATH}")

📂 Racine du projet : /Users/LounesAbd/Etudes/Simplon_DataEng/Projects/Brief-5-RFM-Segmentation-LRZ
📊 Chemin des données : /Users/LounesAbd/Etudes/Simplon_DataEng/Projects/Brief-5-RFM-Segmentation-LRZ/data/online_retail_II.xlsx


In [7]:
# Utilisation de pd.ExcelFile pour ne scanner le fichier qu'une seule fois
with pd.ExcelFile(RAW_DATA_PATH, engine='openpyxl') as xls:
    df_09_10 = pd.read_excel(xls, sheet_name='Year 2009-2010')
    df_10_11 = pd.read_excel(xls, sheet_name='Year 2010-2011')

# Concaténation immédiate
df = pd.concat([df_09_10, df_10_11], ignore_index=True)

print(f"✅ Ingestion réussie : {len(df):,} lignes chargées.")

✅ Ingestion réussie : 1,067,371 lignes chargées.


In [8]:
# Informations générales
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [9]:
# Somme des valeurs nulles par colonne
print("Valeurs manquantes :")
print(df.isnull().sum())

# Pourcentage de Customer ID manquants
missing_cust = (df['Customer ID'].isnull().sum() / len(df)) * 100
print(f"\n⚠️ {missing_cust:.2f}% des lignes n'ont pas de Customer ID.")

Valeurs manquantes :
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

⚠️ 22.77% des lignes n'ont pas de Customer ID.


In [10]:
# Statistiques descriptives
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [11]:
# Compter le nombre d'annulations
cancellations = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Nombre d'annulations détectées : {len(cancellations)}")

Nombre d'annulations détectées : 19494


In [12]:
print(f"Date min : {df['InvoiceDate'].min()}")
print(f"Date max : {df['InvoiceDate'].max()}")

Date min : 2009-12-01 07:45:00
Date max : 2011-12-09 12:50:00


In [13]:
# Nombre de lignes en doublons
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Nombre de doublons exacts : {duplicate_count}")
print(f"Pourcentage de doublons : {duplicate_percentage:.2f}%")

# Visualisation de quelques exemples de doublons pour comprendre leur nature
if duplicate_count > 0:
    print("\nExemples de lignes dupliquées :")
    display(df[df.duplicated(keep=False)].sort_values(by=['Invoice', 'StockCode', 'Customer ID']).head(10))

Nombre de doublons exacts : 34335
Pourcentage de doublons : 3.22%

Exemples de lignes dupliquées :


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [16]:
# 1. Identifier les StockCodes non-standard (lettres)
non_numeric_stocks = df[df['StockCode'].astype(str).str.contains(r'[a-zA-Z]', na=False)]['StockCode'].unique()
print(f"Exemples de StockCodes suspects : {non_numeric_stocks[:10]}")

# 2. Vérifier les prix unitaires à zéro
zero_price = df[df['Price'] == 0]
print(f"Nombre de transactions à prix nul : {len(zero_price)}")

# 3. Vérifier les descriptions suspectes (ex: minuscules)
low_desc = df[df['Description'].str.islower()==True]['Description'].unique()
print(f"Descriptions suspectes repérées : {low_desc}")

Exemples de StockCodes suspects : ['79323P' '79323W' '48173C' '35004B' '84596F' '84596L' '84507B' '84970S'
 '84031A' '84031B']
Nombre de transactions à prix nul : 6202
Descriptions suspectes repérées : ['85123a mixed' 'short' '21733 mixed' 'lost' 'damages' 'invcd as 84879?'
 'sold as gold' 'lost?' 'damaged' 'wet' 'smashed' 'bad quality'
 'discoloured' 'missing' 'missing (wrongly coded?)' 'damaged?' 'damages?'
 'my error - connor' 'wedding co returns?' 'damages, lost bits etc'
 'found' 'lost in space' 'ebay sales' 'temp' 'missing?' 'gone'
 'found again' 'dirty' 'invoice 506647' 'wrong invc' '17129c'
 'phil said so' 'wet and rotting' 'wet & rotting' 'display'
 'wrong ctn size' 'sold as 17003?' 'damages etc' 'mailout'
 'sold in wrong qnty' 'dotcom sales' 'entry error' 'damaged/dirty'
 'dotcom' 'update' 'wet ctn' 'correct previous adjustment'
 'stock credited from royal yacht inc' 'crushed' 'broken' 'display stands'
 'counted' 'check' 'checked' 'cant find' 'rust,missing sets etc'
 'show di

In [17]:
# 1. Conversion en string pour éviter les erreurs de type
stock_series = df['StockCode'].astype(str)

# 2. Application du filtre Regex
pattern = r'^\d{5}[a-zA-Z]$'
mask_suffix = stock_series.str.match(pattern)

# 3. Résultats
df_suffix = df[mask_suffix]
count_unique = df_suffix['StockCode'].nunique()
total_rows = len(df_suffix)

print(f"📊 Nombre de StockCode uniques avec suffixe : {count_unique}")
print(f"📈 Nombre total de lignes concernées : {total_rows}")
print(f"📝 Exemples : {df_suffix['StockCode'].unique()[:10]}")

📊 Nombre de StockCode uniques avec suffixe : 1647
📈 Nombre total de lignes concernées : 127520
📝 Exemples : ['79323P' '79323W' '48173C' '35004B' '84596F' '84596L' '84507B' '84970S'
 '84031A' '84031B']


In [19]:
def analyze_stock_variants(df, target_code):
    # On s'assure de travailler sur des chaînes de caractères
    df['StockCode'] = df['StockCode'].astype(str)
    
    # Extraction de la racine (5 premiers caractères) si on passe un code complet
    root = target_code[:5]
    
    # Filtrage des codes qui commencent par cette racine
    variants = df[df['StockCode'].str.startswith(root)]
    
    # Extraction des suffixes (tout ce qui dépasse les 5 premiers caractères)
    unique_variants = variants['StockCode'].unique()
    
    print(f"🔍 Analyse pour la racine : {root}")
    print(f"🔢 Nombre de variantes trouvées : {len(unique_variants)}")
    print(f"📋 Liste des codes complets : {list(unique_variants)}")
    
    # Optionnel : voir les descriptions pour vérifier s'il s'agit du même produit
    descriptions = variants.groupby('StockCode')['Description'].unique()
    return descriptions

# Test avec votre exemple
analyze_stock_variants(df, "85099B")

🔍 Analyse pour la racine : 85099
🔢 Nombre de variantes trouvées : 6
📋 Liste des codes complets : ['85099B', '85099C', '85099F', '85099b', '85099c', '85099f']


StockCode
85099B    [JUMBO BAG RED WHITE SPOTTY , RED RETROSPOT JU...
85099C                     [JUMBO  BAG BAROQUE BLACK WHITE]
85099F                          [JUMBO BAG STRAWBERRY, nan]
85099b                            [JUMBO BAG RED RETROSPOT]
85099c                     [JUMBO  BAG BAROQUE BLACK WHITE]
85099f                               [JUMBO BAG STRAWBERRY]
Name: Description, dtype: object

In [20]:
# 1. Identification des lignes concernées
df_zero = df[df['Quantity'] == 0]
df_negative = df[df['Quantity'] < 0]

# 2. Statistiques rapides
print(f"🚫 Lignes avec Quantity == 0 : {len(df_zero)}")
print(f"📉 Lignes avec Quantity < 0 : {len(df_negative)}")

# 3. Corrélation avec les annulations (Invoice commençant par 'C')
# On vérifie combien de quantités négatives ne sont PAS des annulations officielles
neg_not_cancelled = df_negative[~df_negative['Invoice'].astype(str).str.startswith('C')]

print(f"⚠️ Quantités négatives sans code 'C' (Anomalies) : {len(neg_not_cancelled)}")

🚫 Lignes avec Quantity == 0 : 0
📉 Lignes avec Quantity < 0 : 22950
⚠️ Quantités négatives sans code 'C' (Anomalies) : 3457


In [23]:
# 1. Identification des prix nuls et négatifs
df_zero_price = df[df['Price'] == 0]
df_negative_price = df[df['Price'] < 0]

# 2. Statistiques
print(f"💸 Lignes avec UnitPrice == 0 : {len(df_zero_price)}")
print(f"📉 Lignes avec UnitPrice < 0 : {len(df_negative_price)}")

# 3. Exploration des causes (Top 10 descriptions pour les prix nuls)
if not df_zero_price.empty:
    print("\n📝 Top Descriptions pour les prix nuls :")
    print(df_zero_price['Description'].value_counts().head(10))

# 4. Zoom sur les prix négatifs (souvent des ajustements comptables)
if not df_negative_price.empty:
    print("\n⚠️ Détails des prix négatifs :")
    display(df_negative_price[['Invoice', 'StockCode', 'Description', 'Price']])

💸 Lignes avec UnitPrice == 0 : 6202
📉 Lignes avec UnitPrice < 0 : 5

📝 Top Descriptions pour les prix nuls :
Description
check                    162
?                         92
damages                   84
damaged                   81
found                     28
missing                   27
sold as set on dotcom     20
Damaged                   17
adjustment                16
OWL DOORSTOP              15
Name: count, dtype: int64

⚠️ Détails des prix négatifs :


,Invoice,StockCode,Description,Price
179403,A506401,B,Adjust bad debt,-53594.36
276274,A516228,B,Adjust bad debt,-44031.79
403472,A528059,B,Adjust bad debt,-38925.87
825444,A563186,B,Adjust bad debt,-11062.06
825445,A563187,B,Adjust bad debt,-11062.06
